# ASSIGNMENT WEEK 7 
# INCREMENTAL DATA PROCESSING USING DELTA LAKE 

## CREATE SAMPLE DATASETS 

### Generate Master Dataset 

In [0]:
import pandas as pd

customer_master = pd.DataFrame({
    "customer_id": [101,102,103,104,105,105,106],
    "name": [
        "Rahul",
        "Priya",
        "Amit",
        "Sneha",
        "Karan",
        "Karan",
        None
    ],
    "city": [
        "Delhi",
        "Mumbai",
        "Pune",
        "Bangalore",
        "Chennai",
        "Chennai",
        "Hyderabad"
    ],
    "email": [
        "rahul@gmail.com",
        "priya@gmail.com",
        "amit@gmail.com",
        "sneha@gmail.com",
        "karan@gmail.com",
        "karan@gmail.com",
        "unknown@gmail.com"
    ]
})

customer_master.to_csv(
    "customer_master.csv",
    index=False
)

print(customer_master)

   customer_id   name       city              email
0          101  Rahul      Delhi    rahul@gmail.com
1          102  Priya     Mumbai    priya@gmail.com
2          103   Amit       Pune     amit@gmail.com
3          104  Sneha  Bangalore    sneha@gmail.com
4          105  Karan    Chennai    karan@gmail.com
5          105  Karan    Chennai    karan@gmail.com
6          106   None  Hyderabad  unknown@gmail.com


### LOAD THE REQUIRED LIBRARIES 

In [0]:
from pyspark.sql import SparkSession
from delta.tables import DeltaTable
from pyspark.sql.functions import *

### LOAD MASTER DATASET 

In [0]:
df_master = spark.read.option("header", True)\
    .csv("file:/databricks/driver/customer_master.csv")

In [0]:
df_master = spark.createDataFrame(customer_master)

display(df_master)

customer_id,name,city,email
101,Rahul,Delhi,rahul@gmail.com
102,Priya,Mumbai,priya@gmail.com
103,Amit,Pune,amit@gmail.com
104,Sneha,Bangalore,sneha@gmail.com
105,Karan,Chennai,karan@gmail.com
105,Karan,Chennai,karan@gmail.com
106,null,Hyderabad,unknown@gmail.com


### Check initial count of the dataset 

In [0]:
print("Original Row Count:", df_master.count())

Original Row Count: 7


## Perform Basic Cleaning 

### Remove Null Values 

In [0]:
df_clean = df_master.na.drop()

In [0]:
display(df_clean)

customer_id,name,city,email
101,Rahul,Delhi,rahul@gmail.com
102,Priya,Mumbai,priya@gmail.com
103,Amit,Pune,amit@gmail.com
104,Sneha,Bangalore,sneha@gmail.com
105,Karan,Chennai,karan@gmail.com
105,Karan,Chennai,karan@gmail.com


### Remove Duplicates 

In [0]:
df_clean = df_clean.dropDuplicates()

In [0]:
display(df_clean)

customer_id,name,city,email
101,Rahul,Delhi,rahul@gmail.com
102,Priya,Mumbai,priya@gmail.com
103,Amit,Pune,amit@gmail.com
104,Sneha,Bangalore,sneha@gmail.com
105,Karan,Chennai,karan@gmail.com


In [0]:
print("Cleaned Row Count:", df_clean.count())

Cleaned Row Count: 5


## Load Cleaned data into Delta table 

### Create Delta Table

In [0]:

df_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("customer_master_delta")

In [0]:

spark.sql("""
SELECT * FROM customer_master_delta
""")

DataFrame[customer_id: bigint, name: string, city: string, email: string]

In [0]:
spark.sql("""
SELECT COUNT(*) FROM customer_master_delta
""").show()

+--------+
|COUNT(*)|
+--------+
|       5|
+--------+



## Create second dataset - Incremental Dataset

In [0]:
incremental_data = [
    (102, "Priya Sharma", "Mumbai", "priya@gmail.com"),
    (104, "Sneha", "Bangalore", "sneha_new@gmail.com"),
    (107, "Rohit", "Gurgaon", "rohit@gmail.com"),
    (108, "Anjali", "Noida", "anjali@gmail.com")
]

columns = [
    "customer_id",
    "name",
    "city",
    "email"
]

df_incremental = spark.createDataFrame(
    incremental_data,
    columns
)

display(df_incremental)

customer_id,name,city,email
102,Priya Sharma,Mumbai,priya@gmail.com
104,Sneha,Bangalore,sneha_new@gmail.com
107,Rohit,Gurgaon,rohit@gmail.com
108,Anjali,Noida,anjali@gmail.com


## Apply MERGE Operation

In [0]:
from delta.tables import DeltaTable

In [0]:
delta_table = DeltaTable.forName(
    spark,
    "customer_master_delta"
)

### merge_operation

In [0]:
delta_table.alias("target") \
.merge(
    df_incremental.alias("source"),
    "target.customer_id = source.customer_id"
) \
.whenMatchedUpdate(
    set={
        "name": "source.name",
        "city": "source.city",
        "email": "source.email"
    }
) \
.whenNotMatchedInsert(
    values={
        "customer_id": "source.customer_id",
        "name": "source.name",
        "city": "source.city",
        "email": "source.email"
    }
) \
.execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

## Validate Results 

### Read final delta table 

In [0]:
final_df = spark.table(
    "customer_master_delta"
)

display(final_df)

customer_id,name,city,email
101,Rahul,Delhi,rahul@gmail.com
103,Amit,Pune,amit@gmail.com
105,Karan,Chennai,karan@gmail.com
102,Priya Sharma,Mumbai,priya@gmail.com
104,Sneha,Bangalore,sneha_new@gmail.com
107,Rohit,Gurgaon,rohit@gmail.com
108,Anjali,Noida,anjali@gmail.com


### Row count validation

In [0]:
print(
    "Final Row Count:",
    final_df.count()
)

Final Row Count: 7


### Duplicate row validation

In [0]:
duplicates = final_df.groupBy(
    "customer_id"
).count().filter(
    "count > 1"
)

duplicates.show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



### Assignment Summary 

In [0]:
print("===== ASSIGNMENT SUMMARY =====")
print("Original Records :", df_master.count())
print("Cleaned Records  :", df_clean.count())
print("Final Records    :", final_df.count())

print("\nUpdated Customers:")
print("102 -> Priya Sharma")
print("104 -> Updated Email")

print("\nInserted Customers:")
print("107 -> Rohit")
print("108 -> Anjali")

===== ASSIGNMENT SUMMARY =====
Original Records : 7
Cleaned Records  : 5
Final Records    : 7

Updated Customers:
102 -> Priya Sharma
104 -> Updated Email

Inserted Customers:
107 -> Rohit
108 -> Anjali
